<a href="https://www.kaggle.com/code/shamanthakreddymallu/s6e6-weighted-blend-meta-stacker?scriptVersionId=328505953" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports & Load

In [ ]:
# PATHS
TRAIN_PATH    = '/kaggle/input/competitions/playground-series-s6e6/train.csv'
TEST_PATH     = '/kaggle/input/competitions/playground-series-s6e6/test.csv'
ORIGINAL_PATH = '/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix,
                             classification_report)
from sklearn.calibration import calibration_curve
from itertools import product
from scipy.optimize import minimize

import warnings
warnings.filterwarnings('ignore')

train    = pd.read_csv(TRAIN_PATH)
test     = pd.read_csv(TEST_PATH)
original = pd.read_csv(ORIGINAL_PATH)

# Standardise original
original = original.rename(columns={'class': 'class'})
original['class'] = original['class'].str.upper().str.strip()

CLASS_COLORS = {'GALAXY': '#4C72B0', 'QSO': '#DD8452', 'STAR': '#55A868'}
CLASS_ORDER  = ['GALAXY', 'QSO', 'STAR']

print('Loaded.')
print(f'Train: {train.shape} | Test: {test.shape} | Original: {original.shape}')

# Clean Original Dataset

In [ ]:
SENTINEL = -9999
SENTINEL_COLS = ['u', 'g', 'z']  # confirmed from EDA (std >> normal range)

# How many rows are affected?
for col in SENTINEL_COLS:
    n = (original[col] < -100).sum()
    print(f'Original {col}: {n:,} sentinel rows ({n/len(original)*100:.1f}%)')

# Remove rows where ANY sentinel is present
mask_clean = ~((original['u'] < -100) | (original['g'] < -100) | (original['z'] < -100))
original_clean = original[mask_clean].copy()
print(f'\nOriginal after cleaning: {len(original_clean):,} rows '
      f'(removed {len(original) - len(original_clean):,})')

# Verify cleaned distributions look sane
print('\nOriginal (cleaned) numeric stats:')
print(original_clean[['u','g','r','i','z','redshift']].describe().T.round(3))

# Feature Engineering

## Feature Engineering Function

In [ ]:
# Fit empirical stellar locus before any feature engineering
# Using raw train STARs in low-redshift zone
_stars  = train[(train['class'] == 'STAR') & (train['redshift'] < 0.15)]
_ug     = _stars['u'] - _stars['g']
_gr     = _stars['g'] - _stars['r']
_coeffs = np.polyfit(_ug, _gr, deg=1)
LOCUS_A, LOCUS_B = float(_coeffs[0]), float(_coeffs[1])
print(f'Stellar locus fitted: g_r = {LOCUS_A:.4f} * u_g + {LOCUS_B:.4f}')

PHOTO_COLS = ['u', 'g', 'r', 'i', 'z']

def engineer_features(df, locus_a=LOCUS_A, locus_b=LOCUS_B):
    df = df.copy()

    # Group 1: Redshift Transforms
    df['log1p_redshift'] = np.log1p(df['redshift'].clip(lower=0))
    df['redshift_sq']    = df['redshift'] ** 2
    df['is_blueshift']   = (df['redshift'] < 0).astype(int)
    df['redshift_zone']  = pd.cut(
        df['redshift'],
        bins=[-np.inf, 0.0, 0.15, 0.50, 1.0, 1.3, np.inf],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)

    # Group 2: Color Indices
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['u_z'] = df['u'] - df['z']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['r_z'] = df['r'] - df['z']

    # Group 3: Magnitude Summary Stats
    df['mean_mag']  = df[PHOTO_COLS].mean(axis=1)
    df['mag_range'] = df[PHOTO_COLS].max(axis=1) - df[PHOTO_COLS].min(axis=1)
    df['mag_std']   = df[PHOTO_COLS].std(axis=1)

    # Group 4: Interaction Features
    df['redshift_x_gr']     = df['redshift'] * df['g_r']
    df['redshift_x_ug']     = df['redshift'] * df['u_g']
    df['redshift_x_uz']     = df['redshift'] * df['u_z']
    df['log_redshift_x_gr'] = df['log1p_redshift'] * df['g_r']

    # Group 5: Positional — redshift-scaled Cartesian
    alpha_rad = np.radians(df['alpha'])
    delta_rad = np.radians(df['delta'])
    r_dist    = df['log1p_redshift']
    df['phys_x'] = r_dist * np.cos(delta_rad) * np.cos(alpha_rad)
    df['phys_y'] = r_dist * np.cos(delta_rad) * np.sin(alpha_rad)
    df['phys_z'] = r_dist * np.sin(delta_rad)

    # Group 6: Stellar Locus + Low-z Zone Features
    df['locus_gr_pred']  = locus_a * df['u_g'] + locus_b
    df['locus_distance'] = df['g_r'] - df['locus_gr_pred']
    df['locus_dist_abs'] = df['locus_distance'].abs()

    return df

train          = engineer_features(train)
test           = engineer_features(test)
original_clean = engineer_features(original_clean)

print('Feature engineering applied.')
print(f'Train shape: {train.shape}')

## Categorical Encoding

In [ ]:
# Ordinal encoding: spectral_type ordered by mean redshift
# Physically meaningful: lower redshift types (cooler/nearer) vs higher (hotter/farther)
st_redshift_order = (
    train.groupby('spectral_type')['redshift']
    .mean()
    .sort_values()
    .index.tolist()
)
print('spectral_type ordered by mean redshift:', st_redshift_order)

st_ordinal_map = {st: i for i, st in enumerate(st_redshift_order)}
print('Ordinal map:', st_ordinal_map)

for df in [train, test]:
    df['spectral_type_ord'] = df['spectral_type'].map(st_ordinal_map)

# Apply same map to original_clean (NaN for any unseen values)
original_clean['spectral_type_ord'] = original_clean.get(
    'spectral_type', pd.Series(dtype=str)
).map(st_ordinal_map) if 'spectral_type' in original_clean.columns else np.nan

# galaxy_population binary encoding 
gp_map = {'Red_Sequence': 1, 'Blue_Cloud': 0}

for df in [train, test]:
    df['galaxy_pop_bin'] = df['galaxy_population'].map(gp_map)

# Original doesn't have this — will be NaN (LightGBM handles natively)
original_clean['galaxy_pop_bin'] = np.nan

# Combined categorical interaction
for df in [train, test]:
    df['spec_gpop'] = df['spectral_type'] + '_' + df['galaxy_population']

# Encode combined label (fit on train, apply to test)
spec_gpop_vals = train['spec_gpop'].unique()
spec_gpop_map  = {v: i for i, v in enumerate(spec_gpop_vals)}
print('\nspec_gpop unique values and encoding:')
for k, v in sorted(spec_gpop_map.items()):
    print(f'  {k}: {v}')

for df in [train, test]:
    df['spec_gpop_enc'] = df['spec_gpop'].map(spec_gpop_map)

# Target encoding — spectral_type
# For each class, encode as P(class | spectral_type) — useful for all 3 classes
for cls in CLASS_ORDER:
    col_name = f'spec_target_{cls.lower()}'
    target_map = train.groupby('spectral_type')['class'].apply(
        lambda x: (x == cls).mean()
    )
    print(f'\nTarget encode spectral_type → P({cls}):')
    print(target_map.round(3))
    for df in [train, test]:
        df[col_name] = df['spectral_type'].map(target_map)

# Target encoding — galaxy_population
for cls in CLASS_ORDER:
    col_name = f'gpop_target_{cls.lower()}'
    target_map = train.groupby('galaxy_population')['class'].apply(
        lambda x: (x == cls).mean()
    )
    print(f'\nTarget encode galaxy_population → P({cls}):')
    print(target_map.round(3))
    for df in [train, test]:
        df[col_name] = df['galaxy_population'].map(target_map)

print('\nCategorical encoding done.')

## Integrate Original Data

In [ ]:
SHARED_NUMERIC = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

ENGINEERED_COLS = [
    'log1p_redshift', 'redshift_sq', 'is_blueshift', 'redshift_zone',
    'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_z', 'g_i', 'g_z', 'r_z',
    'mean_mag', 'mag_range', 'mag_std',
    'redshift_x_gr', 'redshift_x_ug', 'redshift_x_uz', 'log_redshift_x_gr',
    'phys_x', 'phys_y', 'phys_z',
]

CATEGORICAL_ENGINEERED = [
    'spectral_type_ord', 'galaxy_pop_bin', 'spec_gpop_enc',
    'spec_target_galaxy', 'spec_target_qso', 'spec_target_star',
    'gpop_target_galaxy', 'gpop_target_qso', 'gpop_target_star',
]

ALL_FEATURE_COLS = (
    SHARED_NUMERIC
    + ENGINEERED_COLS
    + CATEGORICAL_ENGINEERED
)

# Build original augmentation rows — set missing categorical-derived cols to NaN
orig_aug = original_clean[SHARED_NUMERIC + ['class']].copy()
orig_aug['source'] = 'original'

for col in ENGINEERED_COLS:
    if col in original_clean.columns:
        orig_aug[col] = original_clean[col].values
    else:
        orig_aug[col] = np.nan

for col in CATEGORICAL_ENGINEERED:
    orig_aug[col] = np.nan  # all NaN — no spectral_type / galaxy_population

train['source'] = 'kaggle'
train_augmented = pd.concat([train, orig_aug], ignore_index=True)

print(f'Kaggle train only : {len(train):,}')
print(f'Original (cleaned): {len(orig_aug):,}')
print(f'Augmented train   : {len(train_augmented):,}')
print(f'\nClass distribution in augmented:')
print(train_augmented['class'].value_counts())

## Final Features

In [ ]:
FINAL_FEATURES = [
    'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift',
    'log1p_redshift', 'redshift_sq', 'redshift_zone',
    'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_z', 'g_i', 'g_z', 'r_z',
    'mean_mag', 'mag_range', 'mag_std',
    'redshift_x_gr', 'redshift_x_ug', 'redshift_x_uz', 'log_redshift_x_gr',
    'phys_x', 'phys_y', 'phys_z',
    'spectral_type_ord', 'galaxy_pop_bin', 'spec_gpop_enc',
    'spec_target_galaxy', 'spec_target_qso', 'spec_target_star',
    'gpop_target_galaxy', 'gpop_target_qso', 'gpop_target_star',
    'locus_distance', 'locus_dist_abs',
]
print(f'FINAL_FEATURES: {len(FINAL_FEATURES)} features')

## Sanity Checks

In [ ]:
# Check for any unexpected inf/nan in final features on kaggle train
print('NaN/Inf check — Kaggle Train')
for col in FINAL_FEATURES:
    n_nan = train[col].isna().sum()
    n_inf = np.isinf(train[col].replace([np.inf, -np.inf], np.nan).fillna(0)).sum()
    if n_nan > 0 or n_inf > 0:
        print(f'  {col}: {n_nan} NaN, {n_inf} Inf')
print('  (no output = all clean)')

print('\n NaN/Inf check — Test')
for col in FINAL_FEATURES:
    n_nan = test[col].isna().sum()
    n_inf = np.isinf(test[col].replace([np.inf, -np.inf], np.nan).fillna(0)).sum()
    if n_nan > 0 or n_inf > 0:
        print(f'  {col}: {n_nan} NaN, {n_inf} Inf')
print('  (no output = all clean)')

print(f'\nFinal shapes:')
print(f'  train[FINAL_FEATURES]: {train[FINAL_FEATURES].shape}')
print(f'  test[FINAL_FEATURES] : {test[FINAL_FEATURES].shape}')
print(f'  train_augmented      : {train_augmented.shape}')

# Modeling

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(train['class'])

N_FOLDS = 5
SEED = 42
N_CLASSES = 3

print(f'Class mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Training

In [ ]:
y_aug   = le.transform(train_augmented['class'])
kag_idx = np.where(train_augmented['source'] == 'kaggle')[0]
y_eval  = y_aug[kag_idx]

MODELS = {}
def add(name, oof_path, test_path):
    MODELS[name] = (np.load(oof_path)[kag_idx], np.load(test_path))

add('LGB',  f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-lightgbm-0-96594/oof_lgb.npy',   f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-lightgbm-0-96594/test_lgb.npy')
add('XGB',  f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-xgboost-0-96586/oof_xgb.npy',    f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-xgboost-0-96586/test_xgb.npy')
add('CAT',  f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-catboost-0-96808/oof_cat.npy',   f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-catboost-0-96808/test_cat.npy')
add('DCN',  f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-dcn/oof_dcn.npy',                f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-dcn/test_dcn.npy')
add('TabM', f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-tabm/oof_tabm.npy',              f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-tabm/test_tabm.npy')
add('MLP',  f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-realmlp-0-96611/oof_mlp.npy',    f'/kaggle/input/notebooks/shamanthakreddymallu/s6e6-realmlp-0-96611/test_mlp.npy')
# CD external
#add('XGBCD', f'/kaggle/input/notebooks/cdeotte/xgb-v5-for-s6e6/train_oof/xgb-5_oof.npy',         f'/kaggle/input/notebooks/cdeotte/xgb-v5-for-s6e6/test_preds/xgb-5_test_preds.npy')
#add('CATCD', f'/kaggle/input/notebooks/cdeotte/cat-v3-for-s6e6/train_oof/cat-3_oof.npy',         f'/kaggle/input/notebooks/cdeotte/cat-v3-for-s6e6/test_preds/cat-3_test_preds.npy')
#add('MLPCD', f'/kaggle/input/notebooks/cdeotte/realmlp-v5-for-s6e6/train_oof/realmlp-5_oof.npy', f'/kaggle/input/notebooks/cdeotte/realmlp-v5-for-s6e6/test_preds/realmlp-5_test_preds.npy')

NAMES = list(MODELS)
print(f'Loaded {len(NAMES)} models: {NAMES}')
for n in NAMES:
    o = MODELS[n][0]
    print(f'  {n:6s} OOF BA = {balanced_accuracy_score(y_eval, o.argmax(1)):.5f}  shape={o.shape}')


# Baseline: Weighted Blend (Nelder-Mead)

In [ ]:
from scipy.optimize import minimize

def fit_weights(names):
    stack = [MODELS[n][0] for n in names]
    def neg(w):
        w = np.abs(w); w = w / w.sum()
        return -balanced_accuracy_score(y_eval, sum(wi*p for wi,p in zip(w,stack)).argmax(1))
    res = minimize(neg, np.ones(len(stack))/len(stack), method='Nelder-Mead',
                   options={'xatol':1e-4,'fatol':1e-7,'maxiter':6000})
    w = np.abs(res.x)/np.abs(res.x).sum()
    return -res.fun, w

w_score, w = fit_weights(NAMES)
print('Weighted-blend OOF BA:', round(w_score,5))
for n,wi in zip(NAMES,w):
    print(f'  {n}: {wi:.3f}')

oof_wblend  = sum(wi*MODELS[n][0] for n,wi in zip(NAMES,w))
test_wblend = sum(wi*MODELS[n][1] for n,wi in zip(NAMES,w))


# Meta-Stacker (Level-2)

A meta-learner trained on the OOF probability columns of every base model, with
honest nested CV so the meta-OOF is leak-free. Two meta-learners are tried:
multinomial Logistic Regression (Ridge-style, robust) and a shallow LightGBM
(can learn context-dependent trust, e.g. which model to favour at high redshift).
A few raw context features (redshift, key colors) are appended so the meta-model
can route by object type. Judged against the weighted-blend baseline on OOF.

In [ ]:
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

# ----- base model OOF/test probability columns -----
base_oof  = np.hstack([MODELS[n][0] for n in NAMES]).astype('float32')
base_test = np.hstack([MODELS[n][1] for n in NAMES]).astype('float32')
base_feat_names = [f'{n}_{c}' for n in NAMES for c in ['G','Q','S']]

# ----- expanded context features (axes the meta-model can route trust on) -----
CTX_COLS = ['redshift', 'log1p_redshift', 'redshift_sq', 'redshift_zone',
            'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_z', 'g_i',
            'mean_mag', 'mag_range', 'mag_std',
            'locus_distance', 'locus_dist_abs',
            'redshift_x_gr', 'redshift_x_ug']
CTX_COLS = [c for c in CTX_COLS if c in train_augmented.columns and c in test.columns]

ctx_tr = np.nan_to_num(train_augmented.iloc[kag_idx][CTX_COLS].to_numpy('float32'))
ctx_te = np.nan_to_num(test[CTX_COLS].to_numpy('float32'))
print(f'Context features ({len(CTX_COLS)}): {CTX_COLS}')

X_meta_oof  = np.hstack([base_oof,  ctx_tr])
X_meta_test = np.hstack([base_test, ctx_te])
META_NAMES  = base_feat_names + CTX_COLS
print('Meta X shape:', X_meta_oof.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------- Meta 1: multinomial Logistic Regression ----------
lr_oof  = np.zeros((len(y_eval), N_CLASSES)); lr_test = np.zeros((len(X_meta_test), N_CLASSES))
for tr, va in skf.split(X_meta_oof, y_eval):
    clf = LogisticRegression(max_iter=500, C=2.0, class_weight='balanced')
    clf.fit(X_meta_oof[tr], y_eval[tr])
    lr_oof[va] = clf.predict_proba(X_meta_oof[va])
    lr_test   += clf.predict_proba(X_meta_test) / skf.get_n_splits()
lr_ba = balanced_accuracy_score(y_eval, lr_oof.argmax(1))
print(f'LR stacker   OOF BA: {lr_ba:.5f}')

# ---------- Meta 2: shallow LightGBM (depth 3 — conservative) ----------
lgb_oof  = np.zeros((len(y_eval), N_CLASSES)); lgb_test = np.zeros((len(X_meta_test), N_CLASSES))
imp_gain = np.zeros(X_meta_oof.shape[1])
for tr, va in skf.split(X_meta_oof, y_eval):
    m = lgb.LGBMClassifier(objective='multiclass', num_class=N_CLASSES,
                           n_estimators=400, learning_rate=0.03, max_depth=3,
                           num_leaves=7, subsample=0.8, colsample_bytree=0.8,
                           class_weight='balanced', random_state=42, verbose=-1)
    m.fit(X_meta_oof[tr], y_eval[tr], eval_set=[(X_meta_oof[va], y_eval[va])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_oof[va] = m.predict_proba(X_meta_oof[va])
    lgb_test   += m.predict_proba(X_meta_test) / skf.get_n_splits()
    imp_gain   += m.booster_.feature_importance(importance_type='gain')
lgb_ba = balanced_accuracy_score(y_eval, lgb_oof.argmax(1))
print(f'LGB stacker  OOF BA: {lgb_ba:.5f}')

avg_oof  = (lr_oof + lgb_oof) / 2; avg_test = (lr_test + lgb_test) / 2
avg_ba   = balanced_accuracy_score(y_eval, avg_oof.argmax(1))
print(f'LR+LGB avg   OOF BA: {avg_ba:.5f}')
print(f'\nBaseline weighted blend OOF BA: {w_score:.5f}')

candidates = {'weighted': (w_score, oof_wblend, test_wblend),
              'LR': (lr_ba, lr_oof, lr_test),
              'LGB': (lgb_ba, lgb_oof, lgb_test),
              'LR+LGB': (avg_ba, avg_oof, avg_test)}
best_name = max(candidates, key=lambda k: candidates[k][0])
best_ba, oof_blend, test_blend = candidates[best_name]
print(f'BEST: {best_name}  OOF BA {best_ba:.5f}')

## Feature Importance

In [ ]:
# ---- Stacker feature importance: what did the meta-model learn to trust? ----
imp = pd.Series(imp_gain, index=META_NAMES).sort_values(ascending=False)
is_ctx = imp.index.isin(CTX_COLS)
ctx_imp = imp[is_ctx]

per_model = pd.Series({n: imp[[f'{n}_G', f'{n}_Q', f'{n}_S']].sum() for n in NAMES})
per_model = per_model.sort_values(ascending=False)

print('=== Aggregate trust per base model (%) ===')
print((per_model / imp.sum() * 100).round(1).to_string())
print(f'\n=== Context features total share: {ctx_imp.sum()/imp.sum()*100:.1f}% ===')
print(ctx_imp.round(0).to_string())

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
(per_model / per_model.sum() * 100).plot.barh(ax=ax[0], color='#4C72B0')
ax[0].set_title('Stacker trust per base model (%)', fontweight='bold'); ax[0].invert_yaxis()
ctx_imp.sort_values().plot.barh(ax=ax[1], color='#E87843')
ax[1].set_title('Context feature importance (routing)', fontweight='bold')
plt.tight_layout(); plt.show()

# Class Weights (on chosen stacker output)

In [ ]:
def optimize_class_weights(probs, y_true, coarse=21, refine=21):
    grid       = np.linspace(0.6, 1.6, coarse)
    best_score = balanced_accuracy_score(y_true, probs.argmax(1))
    best_w     = np.ones(3)
    for wq, ws in product(grid, grid):
        w = np.array([1.0, wq, ws])
        s = balanced_accuracy_score(y_true, (probs * w).argmax(1))
        if s > best_score:
            best_score, best_w = s, w
    fq = np.linspace(best_w[1]-0.05, best_w[1]+0.05, refine)
    fs = np.linspace(best_w[2]-0.05, best_w[2]+0.05, refine)
    for wq, ws in product(fq, fs):
        w = np.array([1.0, wq, ws])
        s = balanced_accuracy_score(y_true, (probs * w).argmax(1))
        if s > best_score:
            best_score, best_w = s, w
    return best_w, best_score

global_w, global_score = optimize_class_weights(oof_blend, y_eval)
test_final = test_blend * global_w
test_final = test_final / test_final.sum(axis=1, keepdims=True)
print(f'Class weights : GALAXY={global_w[0]:.3f} QSO={global_w[1]:.3f} STAR={global_w[2]:.3f}')
print(f'Chosen stacker OOF      : {best_ba:.5f}')
print(f'+ class weights OOF     : {global_score:.5f}')
print(f'(baseline weighted blend: {w_score:.5f})')


# Submission

In [ ]:
labels = le.inverse_transform(test_final.argmax(axis=1))
sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv')
sub['class'] = labels
sub.to_csv('submission.csv', index=False)
print('Saved submission.csv  | best stacker:', best_name)
print(pd.Series(labels).value_counts().to_string())
print(f'Final OOF (weighted): {global_score:.5f}')